# Manual Evaluation of Bangla → Chatgaya Translation Model

## Objective

This notebook manually evaluates the fine-tuned mBART-50 translation model.

For each input:

- A Bangla sentence is provided.
- The model generates a Chatgaya translation.
- A reference Chatgaya translation is entered manually.
- BLEU and chrF scores are calculated.

This notebook is useful for qualitative evaluation and demonstration.

In [1]:
# ==========================================================
# Install Required Libraries
# ==========================================================

!pip install -q \
transformers==4.46.3 \
evaluate==0.4.3 \
sentencepiece==0.2.0 \
sacrebleu==2.4.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import torch
import evaluate

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast
)

In [3]:
# ==========================================================
# Mount Google Drive
# ==========================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Load Fine-tuned Model

The fine-tuned mBART-50 model and tokenizer are loaded from Google Drive.

This model will be used for manual translation and evaluation.

In [5]:
# ==========================================================
# Load Fine-tuned Model
# ==========================================================

MODEL_PATH = "/content/drive/MyDrive/Chatgaya-Bangla-NMT/mbart_bn2ctg_new"

tokenizer = MBart50TokenizerFast.from_pretrained(
    MODEL_PATH
)

model = MBartForConditionalGeneration.from_pretrained(
    MODEL_PATH
)

tokenizer.src_lang = "bn_IN"
tokenizer.tgt_lang = "bn_IN"

model.to(device)
model.eval()

print("✅ Model Loaded Successfully")

✅ Model Loaded Successfully


In [6]:
# ==========================================================
# Load BLEU & chrF Metrics
# ==========================================================

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

print("✅ Metrics Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Metrics Loaded Successfully


# Translation Function

This function translates a Bangla sentence into Chatgaya using the fine-tuned model.

In [7]:
# ==========================================================
# Bangla → Chatgaya Translation
# ==========================================================

def translate(sentence):

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            forced_bos_token_id=tokenizer.lang_code_to_id["bn_IN"],

            max_new_tokens=64,

            num_beams=4,

            early_stopping=True
        )

    prediction = tokenizer.batch_decode(

        outputs,

        skip_special_tokens=True

    )[0]

    return prediction

# Calculate BLEU & chrF

This function compares the model prediction with the reference translation and calculates BLEU and chrF scores.

In [8]:
# ==========================================================
# Evaluate One Sentence
# ==========================================================

def evaluate_translation(bangla, reference):

    prediction = translate(bangla)

    bleu_score = bleu.compute(
        predictions=[prediction],
        references=[[reference]]
    )["score"]

    chrf_score = chrf.compute(
        predictions=[prediction],
        references=[reference]
    )["score"]

    print("="*60)

    print("Bangla Sentence")
    print(bangla)

    print()

    print("Reference")
    print(reference)

    print()

    print("Prediction")
    print(prediction)

    print()

    print(f"BLEU : {bleu_score:.2f}")

    print(f"chrF : {chrf_score:.2f}")

In [12]:
bangla_sentence = "আমি আজ বাজারে যাব"

reference_translation = "আঁই আজিয়ে বাজারুত যাইয়্যুম"

evaluate_translation(
    bangla_sentence,
    reference_translation
)

Bangla Sentence
আমি আজ বাজারে যাব

Reference
আঁই আজিয়ে বাজারুত যাইয়্যুম

Prediction
আঁই আজিয়ে বাজারোত যাইয়্যুম

BLEU : 15.97
chrF : 47.28
